In [ ]:
from google.cloud import bigquery, storage
from pyspark.sql import SparkSession
from datetime import datetime, timedelta, timezone
import json

In [ ]:
# Thông tin project
project_id = 'healthcare-project-longnn'

# Khởi tạo kết nối tới Cloud Storage và BigQuery
storage_client = storage.Client()
bq_client = bigquery.Client()

# Cấu hình các file và thư mục configs, temp và landing để làm việc với Cloud Storage
bucket_name = 'healthcare_project_longnn'
temp_folder = 'temp/'
landing_path_a = f"gs://{bucket_name}/landing/hospital_a/"
landing_path_b = f"gs://{bucket_name}/landing/hospital_b/"
archive_path_a = f"gs://{bucket_name}/landing/hospital_a/archive/"
archive_path_b = f"gs://{bucket_name}/landing/hospital_b/archive/"
config_file_path = f"gs://{bucket_name}/configs/load_config.csv"


# Cấu hình đường dẫn trong BigQuery
bq_temp_dataset = 'temp_dataset'
bq_audit_table = f"{project_id}.{bq_temp_dataset}.audit_log"
bq_log_table =  f"{project_id}.{bq_temp_dataset}.pipeline_log"
bq_temp_path = f"{bucket_name}/temp/"

# Cấu hình PostgreSQL trong Cloud SQL
## hospital-a instance
hospital_a_db_config = {
    'url': 'jdbc:postgresql://10.76.192.3:5432/hospital_a',
    'driver': 'org.postgresql.Driver',
    'user': 'longnn28',
    'password': 'Pythongold@64'
}
## hospital-b instance
hospital_b_db_config = {
    'url': 'jdbc:postgresql://10.76.192.5:5432/hospital_b',
    'driver': 'org.postgresql.Driver',
    'user': 'longnn28',
    'password': 'Pythongold@64'
}

# Khởi tạo SparkSession
spark = SparkSession.builder \
    .appName("HospitalDataIngestion") \
    .getOrCreate()

In [ ]:
log_entries = []
# Function logging
def log_event(event_type, message, datasource = None, table = None):
    local_tz = timezone(timedelta(hours=7))
    log_entry = {
        'timestamp': datetime.now(timezone.utc).astimezone(local_tz).isoformat(),
        'event_type': event_type,
        'message': message,
        'datasource': datasource,
        'table': table
    }
    log_entries.append(log_entry)
    print(f"{log_entry['timestamp']} - {log_entry['event_type']}: {log_entry['message']} (Datasource: {log_entry['datasource']}, Table: {log_entry['table']})")


In [ ]:
# Hàm đọc file config từ Cloud Storage
def read_config_file():
    df = spark.read.format("csv").option("header", "true").load(config_file_path)
    log_event("INFO", f"Config file read successfully")
    return df


In [ ]:
# Save logs vào Cloud Storage
def save_logs_to_storage():
    local_tz = timezone(timedelta(hours=7))
    log_file_name = f"pipeline_log_{datetime.now(timezone.utc).astimezone(local_tz).strftime('%Y%m%d_%H%M%S')}.json"
    log_file_path = f"temp/pipeline_logs/{log_file_name}"

    json_data = json.dumps(log_entries, indent=4)

    bucket = storage_client.bucket(bucket_name)
    blob = bucket.blob(log_file_path)

    blob.upload_from_string(json_data, content_type='application/json')

    print(f"Logs saved to {log_file_path} in Cloud Storage")



In [ ]:
# Save logs vào BigQuery
def save_logs_to_bigquery():
    if log_entries:
        log_df = spark.createDataFrame(log_entries)
        log_df.write.format('bigquery') \
            .option('table', bq_log_table) \
            .option('temporaryGcsBucket', bq_temp_path) \
            .mode('append') \
            .save()
        
        print(f"Logs saved to BigQuery table {bq_log_table}")
    else:
        print("No logs to save to BigQuery")


In [ ]:
# Move các file đã xử lý vào thư mục archive
def move_existing_files_to_archive(datasource, table):
    blob = list(storage_client.bucket(bucket_name).list_blobs(prefix=f"landing/{datasource}/{table}/"))
    existing_files = [b.name for b in blob if b.name.endswith('.json')]

    if not existing_files:
        log_event("INFO", f"No existing files to move for {datasource}.{table}", datasource, table)
        return
    
    local_tz = timezone(timedelta(hours=7))
    run_ts = datetime.utcnow().astimezone(local_tz).strftime('%Y%m%d_%H%M%S')
    for file in existing_files:
        source_blob = storage_client.bucket(bucket_name).blob(file)

        # Trích xuất ngày từ tên file
        date_part = file.split('_')[-1].split('.')[0]
        year, month, day = date_part[-4:], date_part[2:4], date_part[:2]
        file_name = file.split('/')[-1]

        # Move tới archive
        archive_path = f"landing/{datasource}/archive/{table}/{year}/{month}/{day}/{run_ts}/{file_name}"
        destination_blob = storage_client.bucket(bucket_name).blob(archive_path)

        # Copy và xóa file gốc
        storage_client.bucket(bucket_name).copy_blob(source_blob, storage_client.bucket(bucket_name), archive_path)
        source_blob.delete()

        log_event("INFO", f"Moved file {file} to archive at {archive_path}", datasource, table)



In [ ]:
# Lấy watermark từ BigQuery Audit Table
def get_latest_watermark(datasource, table):
    query = f"""
        SELECT MAX(watermark) AS latest_watermark
        FROM `{bq_audit_table}`
        WHERE data_source = '{datasource}' AND tablename = '{table}'
    """
    query_job = bq_client.query(query)
    result = query_job.result()
    for row in result:
        return row.latest_watermark if row.latest_watermark else '1900-01-01 00:00:00'
    

In [ ]:
# Trích xuất dữ liệu từ PostgreSQL và lưu vào Cloud Storage
def extract_and_save_to_landing(datasource, table, load_type, watermark_col):
    try:
        last_watermark = get_latest_watermark(datasource, table) if load_type == 'incremental' else None
        log_event("INFO", f"Starting data extraction for {datasource}.{table} with load type {load_type} and last watermark {last_watermark}", datasource, table)

        query = f"(SELECT * FROM {table})" if load_type.lower() == 'full' else f"(SELECT * FROM {table} WHERE {watermark_col} > '{last_watermark}')"
        df = spark.read.format("jdbc") \
            .option("url", hospital_a_db_config['url'] if datasource == 'hospital_a' else hospital_b_db_config['url']) \
            .option("driver", hospital_a_db_config['driver'] if datasource == 'hospital_a' else hospital_b_db_config['driver']) \
            .option("user", hospital_a_db_config['user'] if datasource == 'hospital_a' else hospital_b_db_config['user']) \
            .option("password", hospital_a_db_config['password'] if datasource == 'hospital_a' else hospital_b_db_config['password']) \
            .option("dbtable", query) \
            .load()
        
        log_event("INFO", f"Data extraction completed for {datasource}.{table}", datasource, table)

        local_tz = timezone(timedelta(hours=7))
        today = datetime.now(timezone.utc).astimezone(local_tz).strftime('%d%m%Y')
        json_file_path = f"landing/{datasource}/{table}/{table}_{today}.json"

        bucket = storage_client.bucket(bucket_name)
        blob = bucket.blob(json_file_path)
        blob.upload_from_string(df.toJSON().collect(), content_type='application/json')

        log_event("SUCCESS", f"Data saved to Cloud Storage at {json_file_path}", datasource, table)

        local_tz = timezone(timedelta(hours=7))
        audit_timestamp = datetime.now(timezone.utc).astimezone(local_tz)

        # Thêm entry vào audit log
        audit_df = spark.createDataFrame(
            [(datasource, table, load_type, df.count(), audit_timestamp, "SUCCESS")],
            ["data_source", "tablename", "load_type", "record_count", "load_timestamp", "status"]
        )

        audit_df.write.format('bigquery') \
            .option('table', bq_audit_table) \
            .option('temporaryGcsBucket', bq_temp_path) \
            .mode('append') \
            .save()
        
        log_event("INFO", f"Audit log updated for {datasource}.{table}", datasource, table)

    except Exception as e:
        log_event("ERROR", f"Error processing {datasource}.{table}: {str(e)}", datasource, table)



SyntaxError: incomplete input (2137123886.py, line 39)

In [ ]:
# Xử lý dữ liệu cho từng bảng theo config
config_df = read_config_file()
for row in config_df.collect():
    if row['is_active'] == '1':
        db, src, table, load_type, watermark, _, targetpath = row
        move_existing_files_to_archive(src, table)
        extract_and_save_to_landing(src, table, load_type, watermark)

save_logs_to_storage()
save_logs_to_bigquery()
